In [ ]:
# ═══════════════════════════════════════════════════════════════
# CEL·LA 0 — Executa PRIMER aquesta cel·la abans de cap altra
# Fixa la versió de transformers compatible amb el model AINA
# Model usat: Helsinki-NLP/opus-mt-es-ca (MarianMT natiu, afinable)
# (projecte-aina/aina-translator-es-ca és CTranslate2 i no és afinable directament)
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

print('Fixant versió de transformers i instal·lant dependències crítiques...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install',
     'transformers==4.45.0', 'sentencepiece', 'evaluate', 'sacremoses', '--quiet'],
    capture_output=True
)

import transformers
print(f'✓ Transformers: {transformers.__version__}')
assert transformers.__version__.startswith('4.'), \
    f'ERROR: Versió incorrecta de transformers: {transformers.__version__}. ' \
    f'Executa: pip install transformers==4.45.0'
print('✓ Versió correcta.')

# Verifica que tots els mòduls necessaris estan disponibles
moduls_necessaris = [
    'transformers', 'datasets', 'sacrebleu',
    'sentencepiece', 'evaluate', 'accelerate', 'numpy'
]
errors = []
for modul in moduls_necessaris:
    try:
        __import__(modul)
        print(f'  ✓ {modul}')
    except ImportError:
        errors.append(modul)
        print(f'  ✗ {modul} — NO DISPONIBLE')

if errors:
    print(f'\n⚠ Mòduls que falten: {errors}')
    print('Executa de nou la cel·la 0 o reinicia la sessió.')
else:
    print('\n✓ Tots els mòduls disponibles. Continua amb la cel·la 1.')

# Afinament motor TAN SLPL·UV
## Model base: Helsinki-NLP/opus-mt-es-ca → afinar al domini UV
### Servei de Llengües i Política Lingüística · Universitat de València

> ⚠️ **IMPORTANT**: Executa PRIMER la cel·la 0 (fixació de transformers)
> abans de qualsevol altra. Després executa les cel·les en ordre de dalt a baix.
> Si hi ha errors, fes: **Entorn d'execució → Reinicia la sessió** i torna a començar.

### Configuració prèvia a Google Colab
1. Menú: **Entorn d'execució → Canvia el tipus d'entorn → GPU (T4)**
2. Executa la **cel·la 0** per fixar la versió de `transformers`
3. Munta Google Drive i puja la carpeta del corpus a `MyDrive/SLPL_TAN/corpus/`
4. Executa les cel·les en ordre

### Configuració prèvia a Kaggle
1. Activa GPU: **Settings → Accelerator → GPU T4 x2**
2. Puja els fitxers del corpus com a Dataset de Kaggle
3. Executa les cel·les en ordre

In [ ]:
import os
import sys
import pathlib

# ── Detecció robusta de l'entorn ─────────────────────────────────────────────
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

IS_KAGGLE = os.path.exists('/kaggle') and not IS_COLAB
print(f'Entorn detectat: {"Google Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else "Local"}')

# ── Muntatge de Drive (Colab) ────────────────────────────────────────────────
if IS_COLAB:
    from google.colab import drive
    print('Muntant Google Drive...')
    print("⚠ Si apareix una finestra emergent, autoritza l'accés al teu compte Google.")
    try:
        drive.mount('/content/drive', force_remount=True)
        print('✓ Google Drive muntat correctament.')
    except Exception as e:
        print(f'⚠ Error muntant Drive: {e}')
        print("Solució: Menú → Entorn d'execució → Reinicia la sessió, i torna a executar.")
        raise

    CORPUS_DIR = '/content/drive/MyDrive/SLPL_TAN/corpus'
    OUTPUT_DIR = '/content/drive/MyDrive/SLPL_TAN/model-afinar'

    # Verifica que la carpeta del corpus existeix
    if not os.path.exists(CORPUS_DIR):
        print(f'⚠ La carpeta del corpus no existeix: {CORPUS_DIR}')
        print('Crea la carpeta a Google Drive i puja-hi els fitxers:')
        print('  train.es, train.ca, val.es, val.ca')
        raise FileNotFoundError(f'Carpeta no trobada: {CORPUS_DIR}')
    else:
        fitxers = os.listdir(CORPUS_DIR)
        print(f'✓ Carpeta del corpus trobada. Fitxers: {fitxers}')

elif IS_KAGGLE:
    CORPUS_DIR = '/kaggle/input/slpl-corpus'
    OUTPUT_DIR = '/kaggle/working/model-afinar'
else:
    CORPUS_DIR = r'C:\Users\santi\OneDrive\Documents\SLPL\taneu\corpus d entrenament i afinament\processed'
    OUTPUT_DIR = r'C:\Users\santi\OneDrive\Documents\SLPL\taneu\model-afinar'

pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f'Corpus:  {CORPUS_DIR}')
print(f'Sortida: {OUTPUT_DIR}')

In [ ]:
# Instal·la les dependències necessàries
# (Colab ja té PyTorch i CUDA instal·lats)
import subprocess
import sys

packages = [
    'transformers==4.45.0',
    'datasets',
    'sacrebleu',
    'sentencepiece',
    'sacremoses',
    'evaluate',
    'accelerate',
]

print('Instal·lant dependències...')
for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '--quiet'],
        capture_output=True, text=True
    )
    status = '✓' if result.returncode == 0 else '✗'
    print(f'  {status} {pkg}')

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)
print('✓ Totes les dependències instal·lades.')
print('⚠ Si hi ha errors d\'importació, fes: Entorn d\'execució → Reinicia la sessió')

In [ ]:
import torch

cuda_ok = torch.cuda.is_available()
print(f'CUDA disponible: {cuda_ok}')
if cuda_ok:
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
    DEVICE = 'cuda'
    FP16   = True
    BATCH  = 16   # T4 té 16 GB → batch gran
else:
    print('\u26a0 Sense GPU. Entrenament per CPU (molt lent).')
    DEVICE = 'cpu'
    FP16   = False
    BATCH  = 4

In [ ]:
import re
import csv
import random
from pathlib import Path

def carrega_txt_paral_lel(path_es, path_ca):
    pairs = []
    try:
        with open(path_es, encoding='utf-8') as fe, \
             open(path_ca, encoding='utf-8') as fc:
            for s, t in zip(fe, fc):
                s, t = s.strip(), t.strip()
                if s and t:
                    pairs.append((s, t))
    except Exception as e:
        print(f'  ⚠ Error llegint {path_es.name}: {e}')
    return pairs

def carrega_tsv(path):
    pairs = []
    try:
        with open(path, encoding='utf-8') as f:
            reader = csv.reader(f, delimiter='\t')
            next(reader, None)  # salta capçalera si n'hi ha
            for row in reader:
                if len(row) >= 2 and row[0].strip() and row[1].strip():
                    pairs.append((row[0].strip(), row[1].strip()))
    except Exception as e:
        print(f'  ⚠ Error llegint {path.name}: {e}')
    return pairs

def es_valid(src, tgt, min_tok=3, max_tok=120):
    s_tok = src.split()
    t_tok = tgt.split()
    if not (min_tok <= len(s_tok) <= max_tok): return False
    if not (min_tok <= len(t_tok) <= max_tok): return False
    if src.strip() == tgt.strip(): return False
    if sum(c.isalpha() for c in src) / max(len(src), 1) < 0.4: return False
    return True

corpus_path = Path(CORPUS_DIR)
all_pairs   = []
seen        = set()

# ── Opció A: fitxers train.es/train.ca ja processats (prioritat) ─────────────
train_es = corpus_path / 'train.es'
train_ca = corpus_path / 'train.ca'
val_es   = corpus_path / 'val.es'
val_ca   = corpus_path / 'val.ca'

if train_es.exists() and train_ca.exists():
    print('Carregant fitxers train/val preprocessats...')
    train_pairs = carrega_txt_paral_lel(train_es, train_ca)
    val_pairs   = carrega_txt_paral_lel(val_es, val_ca) if val_es.exists() else []
    print(f'  train: {len(train_pairs)} parells')
    print(f'  val:   {len(val_pairs)} parells')

    # Valida els parells
    train_pairs = [(s, t) for s, t in train_pairs if es_valid(s, t)]
    val_pairs   = [(s, t) for s, t in val_pairs   if es_valid(s, t)]

else:
    # ── Opció B: fitxers TSV ──────────────────────────────────────────────────
    print('Cercant fitxers TSV...')
    for f in corpus_path.glob('**/*.tsv'):
        pairs = carrega_tsv(f)
        print(f'  TSV: {f.name} → {len(pairs)} parells')
        all_pairs.extend(pairs)

    # ── Opció C: fitxers .es / .ca paral·lels ─────────────────────────────────
    for f_es in corpus_path.glob('**/*.es'):
        f_ca = f_es.with_suffix('.ca')
        if f_ca.exists():
            pairs = carrega_txt_paral_lel(f_es, f_ca)
            print(f'  PAR: {f_es.name} → {len(pairs)} parells')
            all_pairs.extend(pairs)

    # Neteja i deduplicació
    clean_pairs = []
    for src, tgt in all_pairs:
        if not es_valid(src, tgt): continue
        key = f'{src.lower()}|||{tgt.lower()}'
        if key in seen: continue
        seen.add(key)
        clean_pairs.append((src, tgt))

    random.seed(42)
    random.shuffle(clean_pairs)
    cut         = int(len(clean_pairs) * 0.9)
    train_pairs = clean_pairs[:cut]
    val_pairs   = clean_pairs[cut:]

print(f'\n{"="*45}')
print(f'  CORPUS LLEST')
print(f'{"="*45}')
print(f'  Parells train: {len(train_pairs)}')
print(f'  Parells val:   {len(val_pairs)}')
print(f'{"="*45}')

if len(train_pairs) < 100:
    print('\n⚠ ATENCIÓ: Corpus molt petit o no s\'han trobat fitxers.')
    print(f'  Verifica que la carpeta {CORPUS_DIR} conté:')
    print('    train.es, train.ca (o fitxers .tsv)')
    fitxers_trobats = list(corpus_path.iterdir()) if corpus_path.exists() else []
    print(f'  Fitxers trobats: {[f.name for f in fitxers_trobats[:20]]}')

In [ ]:
MODEL_ID = 'Helsinki-NLP/opus-mt-es-ca'

assert MODEL_ID is not None and isinstance(MODEL_ID, str), \
    f'ERROR: MODEL_ID no és vàlid: {MODEL_ID!r}'
print(f'Model ID: {MODEL_ID}')

from transformers import MarianMTModel, MarianTokenizer

print(f'Carregant model: {MODEL_ID}')
print('(pot trigar 1-2 minuts la primera vegada...)')

tokenizer = MarianTokenizer.from_pretrained(MODEL_ID)
model     = MarianMTModel.from_pretrained(MODEL_ID)

params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'✓ Model carregat. Paràmetres: {params:.1f}M')

# Prova de traducció base (abans de l'afinament)
test = 'El servicio de lenguas ofrece asesoramiento lingüístico a la comunidad universitaria.'
inputs = tokenizer([test], return_tensors='pt', padding=True)
output = model.generate(**inputs)
print(f'\nProva base:')
print(f'  ES: {test}')
print(f'  CA: {tokenizer.decode(output[0], skip_special_tokens=True)}')

In [ ]:
from datasets import Dataset

def preprocess(examples):
    sources = [ex['es'] for ex in examples['translation']]
    targets = [ex['ca'] for ex in examples['translation']]
    model_inputs = tokenizer(
        sources, max_length=256, truncation=True, padding='max_length'
    )
    # text_target= substitueix l'API deprecada as_target_tokenizer()
    # (vàlid des de transformers 4.21+)
    labels = tokenizer(
        text_target=targets,
        max_length=256,
        truncation=True,
        padding='max_length'
    )
    model_inputs['labels'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels['input_ids']
    ]
    return model_inputs

train_dataset = Dataset.from_dict({'translation': [{'es': s, 'ca': t} for s, t in train_pairs]})
val_dataset   = Dataset.from_dict({'translation': [{'es': s, 'ca': t} for s, t in val_pairs]})

print('Tokenitzant corpus...', end=' ')
train_tok = train_dataset.map(preprocess, batched=True, num_proc=2, remove_columns=['translation'])
val_tok   = val_dataset.map(preprocess,   batched=True, num_proc=2, remove_columns=['translation'])
train_tok.set_format('torch')
val_tok.set_format('torch')
print('fet.')
print(f'\u2713 Datasets preparats: {len(train_tok)} train | {len(val_tok)} val')

In [ ]:
# ── Verificació prèvia a l'entrenament ───────────────────────────────────────
print("Verificació prèvia a l'entrenament:")
print(f'  Dispositiu:    {DEVICE}')
print(f'  FP16:          {FP16}')
print(f'  Batch size:    {BATCH}')
print(f'  Parells train: {len(train_tok)}')
print(f'  Parells val:   {len(val_tok)}')
print(f'  Sortida:       {OUTPUT_DIR}')

import torch
if DEVICE == 'cuda':
    mem       = torch.cuda.get_device_properties(0).total_memory / 1024**3
    mem_usada = torch.cuda.memory_allocated(0) / 1024**3
    print(f'  VRAM total:    {mem:.1f} GB')
    print(f'  VRAM usada:    {mem_usada:.1f} GB')
    print(f'  VRAM lliure:   {mem - mem_usada:.1f} GB')

if len(train_tok) < 100:
    print('\n⚠ ATENCIÓ: El corpus és massa petit per entrenar.')
    print('  Verifica que els fitxers s\'han carregat correctament a la cel·la 5.')
else:
    temps_estimat = len(train_tok) / BATCH * 3 / 3600
    print(f'\n  Temps estimat: ~{temps_estimat:.1f} hores')
    print('\n✓ Tot llest. Executa la cel·la següent per iniciar l\'entrenament.')

In [ ]:
import evaluate
import numpy as np
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

metric = evaluate.load('sacrebleu')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = metric.compute(
        predictions=[p.strip() for p in decoded_preds],
        references=[[l.strip()] for l in decoded_labels]
    )
    return {'bleu': round(result['score'], 2)}

training_args = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = BATCH,
    per_device_eval_batch_size  = BATCH,
    gradient_accumulation_steps = 2,
    warmup_steps                = 200,
    weight_decay                = 0.01,
    learning_rate               = 5e-5,
    fp16                        = FP16,
    predict_with_generate       = True,
    generation_max_length       = 256,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'bleu',
    greater_is_better           = True,
    logging_steps               = 50,
    report_to                   = 'none',
    dataloader_pin_memory       = False,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=train_tok, eval_dataset=val_tok,
    tokenizer=tokenizer, data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f'Iniciant afinament: {training_args.num_train_epochs} epochs')
print(f'Corpus: {len(train_tok)} frases train | {len(val_tok)} val')
print(f'Batch size: {BATCH} | FP16: {FP16} | Device: {DEVICE}')
print('\u2500' * 50)

train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f'\n\u2713 Afinament completat.')
print(f'  Pèrdua final: {train_result.training_loss:.4f}')
print(f'  Model guardat a: {OUTPUT_DIR}')

In [ ]:
import sacrebleu as sb
from transformers import pipeline

translator = pipeline(
    'translation',
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    device=0 if DEVICE == 'cuda' else -1,
    max_length=256
)

val_src = [p[0] for p in val_pairs]
val_ref = [p[1] for p in val_pairs]

print('Generant traduccions de validació...', end=' ')
translations = [t['translation_text'] for t in translator(val_src, batch_size=32)]
print('fet.')

bleu = sb.corpus_bleu(translations, [val_ref])
chrf = sb.corpus_chrf(translations, [val_ref])

print(f'\n{"="*45}')
print(f'  RESULTATS FINALS')
print(f'{"="*45}')
print(f'  BLEU:  {bleu.score:.2f}')
print(f'  ChrF:  {chrf.score:.2f}')
print(f'{"="*45}')

print('\nExemples de traducció del model afinar:')
for i in range(min(5, len(val_src))):
    print(f'\n  [{i+1}] ES:  {val_src[i]}')
    print(f'       CA:  {translations[i]}')
    print(f'  REF:      {val_ref[i]}')

In [ ]:
# Descarrega el model afinar al teu ordinador
import zipfile, os

zip_name = '/content/opus-mt-es-ca-afinar-SLPL-UV.zip' if IS_COLAB else '/kaggle/working/opus-mt-es-ca-afinar-SLPL-UV.zip'

print(f'Comprimint el model...', end=' ')
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for file_path in pathlib.Path(OUTPUT_DIR).rglob('*'):
        if file_path.is_file():
            zf.write(file_path, file_path.relative_to(OUTPUT_DIR))

size_mb = os.path.getsize(zip_name) / 1024**2
print(f'fet. ({size_mb:.0f} MB)')

if IS_COLAB:
    from google.colab import files
    print('Iniciant descàrrega...')
    files.download(zip_name)
else:
    print(f'Model disponible a: {zip_name}')
    print('Descarrega\'l des del panell de fitxers de Kaggle.')

print(f'\nDesprés de descomprimir, copia la carpeta a:')
print(f'  C:\\Users\\santi\\OneDrive\\Documents\\SLPL\\taneu\\model-afinar\\')